# Migrate from Chat Completions to the Responses API

The Responses API is the current default surface for the OpenAI platform. It keeps everything
Chat Completions could do, and adds server-side conversation state, native tools, and better
cache behavior on reasoning models.

Chat Completions is **not deprecated** and existing code will keep working. But new platform
capabilities land on Responses first, so most teams eventually want to move.

This guide is the mechanical part of that move: what each parameter becomes, what has no
equivalent, and which shape changes will silently break a port.

**What you'll cover**

1. The smallest possible migration
2. Reading the output (`choices[0].message.content` → `output_text`)
3. A parameter map derived from the installed SDK, not from memory
4. System prompts → `instructions`
5. Multi-turn state → `previous_response_id`
6. Structured outputs — *the shape changes here*
7. Function calling and tools
8. Streaming — from raw deltas to typed events
9. Reasoning models
10. Parameters with no Responses equivalent, and what to do instead

**Prerequisites:** an OpenAI API key and `openai>=3.0`.

In [ ]:
%pip install --upgrade openai --quiet

In [ ]:
import json
import os

from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Any current model works with both APIs; swap this for the model you already use.
MODEL = "gpt-5"

## 1. The smallest possible migration

Three renames cover the majority of simple calls:

| Chat Completions | Responses |
| --- | --- |
| `client.chat.completions.create` | `client.responses.create` |
| `messages=[...]` | `input=[...]` |
| `response.choices[0].message.content` | `response.output_text` |

The message array you already have is accepted as-is by `input`, so the first port usually
needs no restructuring at all.

In [ ]:
# Before — Chat Completions
chat = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a one-line haiku about migrating APIs."}],
)
print(chat.choices[0].message.content)

In [ ]:
# After — Responses. Same message array, new entry point.
resp = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": "Write a one-line haiku about migrating APIs."}],
)
print(resp.output_text)

`input` also accepts a bare string, which is handy for single-turn calls:

In [ ]:
resp = client.responses.create(model=MODEL, input="Write a one-line haiku about migrating APIs.")
print(resp.output_text)

## 2. Reading the output

This is the single most common source of breakage, because Chat Completions returns a list of
*choices* while Responses returns a list of *items*.

`output_text` is a convenience accessor that concatenates the text parts of the output. It is
the direct replacement for `choices[0].message.content` — but it is not the whole response.
When a model calls a tool or emits reasoning, those arrive as separate items in `resp.output`,
and `output_text` will be empty for a turn that only calls a tool.

> **Migration trap:** code that assumes `output_text` is always non-empty will break the first
> time the model decides to call a tool. Check `resp.output` when tools are in play.

In [ ]:
resp = client.responses.create(model=MODEL, input="Name three primary colors.")

print("output_text:", resp.output_text)
print("\nitem types in resp.output:", [item.type for item in resp.output])
print("status:", resp.status)
print("usage:", resp.usage.input_tokens, "in /", resp.usage.output_tokens, "out")

Note the usage field names change too: `prompt_tokens`/`completion_tokens` become
`input_tokens`/`output_tokens`. Any cost-tracking or logging code needs updating.

## 3. The parameter map, derived from the SDK

Rather than trusting a table that goes stale, derive the differences from the SDK you actually
have installed. Run this against your own version to see exactly what applies to you.

In [ ]:
import inspect

from openai.resources.chat.completions import Completions
from openai.resources.responses import Responses

IGNORE = {"self", "extra_headers", "extra_query", "extra_body", "timeout"}

def params(fn):
    return {p for p in inspect.signature(fn).parameters if p not in IGNORE}

chat_params = params(Completions.create)
resp_params = params(Responses.create)

print(f"chat.completions.create : {len(chat_params)} parameters")
print(f"responses.create        : {len(resp_params)} parameters")
print()
print("Chat-only (need a migration decision):")
print(" ", ", ".join(sorted(chat_params - resp_params)))
print()
print("Responses-only (new capabilities):")
print(" ", ", ".join(sorted(resp_params - chat_params)))

Every chat-only parameter falls into one of three buckets. Here is the full disposition:

### Renamed — mechanical change

| Chat Completions | Responses | Note |
| --- | --- | --- |
| `messages` | `input` | accepts the same array, or a plain string |
| `max_tokens` / `max_completion_tokens` | `max_output_tokens` | |
| `response_format` | `text.format` | **shape also changes** — see section 6 |
| `reasoning_effort` | `reasoning.effort` | |
| `verbosity` | `text.verbosity` | `"low"`, `"medium"`, `"high"` |
| `web_search_options` | `tools=[{"type": "web_search"}]` | now a normal tool |
| `functions` / `function_call` | `tools` / `tool_choice` | already legacy in Chat Completions |

### No equivalent — needs a rewrite

| Parameter | What to do |
| --- | --- |
| `n` | issue N separate requests (see section 10) |
| `seed` | no equivalent; pin behavior with evals rather than sampling parameters |
| `logit_bias` | express the constraint in the prompt, or use structured outputs |
| `frequency_penalty` | prompt for concision, or post-process |
| `presence_penalty` | as above |
| `stop` | trim the output yourself, or constrain via structured outputs |
| `logprobs` | use `top_logprobs` plus `include=["message.output_text.logprobs"]` |
| `prediction` | predicted outputs are Chat-Completions-only |
| `audio` / `modalities` | use the audio models or the Realtime API |

### New in Responses — no Chat Completions counterpart

`previous_response_id`, `conversation`, `instructions`, `store`, `background`,
`include`, `truncation`, `max_tool_calls`, `context_management`, `prompt`.

## 4. System prompts become `instructions`

A system/developer message still works inside `input`. But Responses gives system guidance a
dedicated top-level parameter, which keeps it out of the conversation array and makes it
unambiguous when you start chaining turns.

> **Important:** `instructions` applies to *this request only*. It is **not** carried forward
> when you chain with `previous_response_id` — you must pass it again on every call. This
> catches people out when they migrate a system prompt and it silently stops applying on turn 2.

In [ ]:
# Before — system message inside the array
chat = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You answer in exactly one sentence."},
        {"role": "user", "content": "What is the Responses API?"},
    ],
)
print(chat.choices[0].message.content)

In [ ]:
# After — hoisted to instructions
resp = client.responses.create(
    model=MODEL,
    instructions="You answer in exactly one sentence.",
    input="What is the Responses API?",
)
print(resp.output_text)

## 5. Multi-turn conversations

With Chat Completions you kept the whole history client-side and resent it every turn. That
still works with Responses. But Responses can hold the state for you: pass
`previous_response_id` and the server links the new turn to the prior one.

This requires `store=True` (the default). If you send `store=False`, there is nothing on the
server to chain from.

> **Cost caveat, straight from the docs:** *"Previous input tokens in the response chain are
> still billed as input tokens."* Chaining saves you from resending the history over the wire —
> it does not make the context free. Long chains cost the same as long message arrays.

In [ ]:
# Before — you own the history
history = [{"role": "user", "content": "My favourite colour is blue."}]
first = client.chat.completions.create(model=MODEL, messages=history)
history.append({"role": "assistant", "content": first.choices[0].message.content})
history.append({"role": "user", "content": "What did I just tell you?"})
second = client.chat.completions.create(model=MODEL, messages=history)
print(second.choices[0].message.content)

In [ ]:
# After — the server owns the history
first = client.responses.create(model=MODEL, input="My favourite colour is blue.")
second = client.responses.create(
    model=MODEL,
    input="What did I just tell you?",
    previous_response_id=first.id,
)
print(second.output_text)

If you would rather keep managing history yourself — for audit, redaction, or because you
store it in your own database — simply keep doing what you already do and pass the array to
`input`. Server-side state is an option, not a requirement.

## 6. Structured outputs — the shape changes

This is the migration step most likely to fail at runtime, because the rename from
`response_format` to `text.format` is **not** a straight move of the same object.

Chat Completions nests the schema under a `json_schema` key. Responses **flattens it**:
`name`, `schema`, and `strict` sit at the same level as `type`. Moving the old object across
unchanged produces a validation error.

`schema` is also required in Responses, where Chat Completions treated it as optional.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "country": {"type": "string"},
    },
    "required": ["city", "country"],
    "additionalProperties": False,
}

# Before — nested under "json_schema"
chat = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Where is the Eiffel Tower?"}],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "location", "schema": schema, "strict": True},
    },
)
print(json.loads(chat.choices[0].message.content))

In [ ]:
# After — name/schema/strict hoisted alongside "type"
resp = client.responses.create(
    model=MODEL,
    input="Where is the Eiffel Tower?",
    text={
        "format": {
            "type": "json_schema",
            "name": "location",
            "schema": schema,
            "strict": True,
        }
    },
)
print(json.loads(resp.output_text))

If you use the SDK's parsing helper, the equivalent of `chat.completions.parse` is
`responses.parse`, which accepts a Pydantic model via `text_format` and returns it on
`output_parsed`.

## 7. Function calling and tools

Two differences matter. First, the tool definition is **flattened** — Chat Completions nests
the definition under a `function` key, Responses puts `name`/`parameters` at the top level.
Second, tool calls come back as *items in `output`*, not as `message.tool_calls`.

In [ ]:
# Before — nested under "function"
chat_tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
}]

# After — flattened
resp_tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get the weather for a city.",
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
        "additionalProperties": False,
    },
}]

resp = client.responses.create(
    model=MODEL,
    input="What is the weather in Paris?",
    tools=resp_tools,
)

for item in resp.output:
    print(item.type)
    if item.type == "function_call":
        print("  name:", item.name)
        print("  args:", item.arguments)
        print("  call_id:", item.call_id)

Returning the result also changes. Chat Completions wanted a message with
`role="tool"` and a `tool_call_id`. Responses wants a `function_call_output` item keyed by
`call_id`:

In [ ]:
call = next((i for i in resp.output if i.type == "function_call"), None)

if call is None:
    # The model answered directly instead of calling the tool. Nothing to return.
    print("No function_call in output; nothing to send back.")
else:
    follow_up = client.responses.create(
        model=MODEL,
        previous_response_id=resp.id,
        input=[{
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps({"temperature_c": 18, "conditions": "cloudy"}),
        }],
        tools=resp_tools,
    )
    print(follow_up.output_text)

Hosted tools are the real payoff here. Web search, file search, code interpreter, image
generation and MCP servers are all just entries in the same `tools` list, and the platform runs
them for you — no client-side execution loop:

```python
tools=[{"type": "web_search"}]
```

## 8. Streaming

Chat Completions streams opaque chunks that you reassemble by reaching into
`chunk.choices[0].delta.content` and skipping `None`s. Responses streams **typed, named**
events, so you switch on `event.type` instead of defensively probing a nested structure.

In [ ]:
# Before
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Count to five."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")
print()

In [ ]:
# After
stream = client.responses.create(
    model=MODEL,
    input="Count to five.",
    stream=True,
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")
    elif event.type == "response.completed":
        print("\n[done]")

Useful event types while migrating: `response.created`, `response.output_text.delta`,
`response.output_item.added`, `response.function_call_arguments.delta`, `response.completed`.
Print `event.type` in a loop once to see the full sequence for your call.

## 9. Reasoning models

`reasoning_effort` moves under `reasoning`. The more consequential change is that Responses
can persist reasoning state between turns: when you chain with `previous_response_id`, the
model's reasoning items carry forward instead of being discarded and rebuilt.

That is where the documented gains come from — OpenAI reports a **3% improvement on SWE-bench**
and **40–80% better cache utilization** versus the same model on Chat Completions.

In [ ]:
resp = client.responses.create(
    model=MODEL,
    input="A farmer has 17 sheep. All but 9 run away. How many are left?",
    reasoning={"effort": "low"},
)
print(resp.output_text)
print("\nreasoning tokens:", resp.usage.output_tokens_details.reasoning_tokens)

## 10. Parameters with no equivalent

Five sampling parameters simply do not exist in Responses: `n`, `seed`, `logit_bias`,
`frequency_penalty` and `presence_penalty`. If your code depends on them, it needs a rewrite
rather than a rename.

`n` is the most common. Replace one N-way request with N concurrent requests:

In [ ]:
import asyncio

from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

async def sample(prompt, n=3):
    """Chat Completions n=3 equivalent: three concurrent requests."""
    tasks = [
        async_client.responses.create(model=MODEL, input=prompt)
        for _ in range(n)
    ]
    results = await asyncio.gather(*tasks)
    return [r.output_text for r in results]

for i, text in enumerate(await sample("Give me a two-word band name."), 1):
    print(f"{i}. {text}")

The trade-off is real and worth stating: N requests cost N times the input tokens, whereas
`n` billed the prompt once. If you were using `n` for cheap sampling at scale, budget for that
before migrating.

Note also that `temperature` and `top_p` exist in Responses but are not accepted by every
model — reasoning models in particular reject or ignore them. Check the model's reference
page before relying on them.

For the other four, the honest answer is that they were always blunt instruments. Prefer
structured outputs to constrain format, and evals to pin behavior, rather than trying to
reconstruct penalty-based sampling.

## Migration checklist

Work through this against your own codebase:

- [ ] `client.chat.completions.create` → `client.responses.create`
- [ ] `messages=` → `input=`
- [ ] `choices[0].message.content` → `output_text` — and handle the empty case when tools fire
- [ ] `usage.prompt_tokens` / `completion_tokens` → `input_tokens` / `output_tokens`
- [ ] `max_tokens` → `max_output_tokens`
- [ ] `response_format` → `text.format`, **flattening** the `json_schema` nesting
- [ ] tool definitions **flattened** — drop the `function` wrapper
- [ ] tool results → `function_call_output` items keyed by `call_id`
- [ ] streaming → switch on `event.type`
- [ ] system prompt → `instructions`, and re-send it on every chained call
- [ ] audit for `n`, `seed`, `logit_bias`, `frequency_penalty`, `presence_penalty`
- [ ] decide: keep client-side history, or adopt `previous_response_id`

Run the SDK-diff cell in section 3 against your installed version to confirm nothing has
shifted since this guide was written.

### Further reading

- [Migrate to the Responses API](https://developers.openai.com/api/docs/guides/migrate-to-responses)
- [Responses API reference](https://developers.openai.com/api/docs/api-reference/responses)
- [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state)